# Faster R-CNN on RTSD — Modified ResNet50-FPN

The detector uses compatible COCO-pretrained backbone weights. `BatchNorm2d` and `FrozenBatchNorm2d` are replaced with `GroupNorm` (32 groups), and `ReLU` is replaced with `SiLU`. GroupNorm parameters are initialized separately; the RPN and detection head are built separately.

*Implementation reconstructed from the thesis description. Outputs are cleared; this revised notebook has not been executed.*

## 1. Imports

In [ ]:
import os
import json
import numpy as np
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import defaultdict, Counter
import torchvision.transforms.functional as F
import time

print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Device: {device}")

## 2. Paths and hyperparameters

Set `DATASET_ROOT` to the RTSD data directory.

In [ ]:
import os

# RTSD dataset root (adjust for your environment).
DATASET_ROOT = 'rtsd_data'

TRAIN_ANNO  = os.path.join(DATASET_ROOT, 'train_anno.json')
VAL_ANNO    = os.path.join(DATASET_ROOT, 'val_anno.json')
LABEL_MAP   = os.path.join(DATASET_ROOT, 'label_map.json')
FRAMES_DIR  = os.path.join(DATASET_ROOT, 'rtsd-frames', 'rtsd-frames')


# Training hyperparameters
MIN_INSTANCES_PER_CLASS = 50
BATCH_SIZE  = 2
NUM_EPOCHS  = 15
LR          = 0.005
MOMENTUM    = 0.9
WEIGHT_DECAY = 0.0005

print("Dataset paths configured.")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Frames directory exists: {os.path.exists(FRAMES_DIR)}")

# If missing, check the extracted directory structure.


## 3. Dataset inspection and class filtering

In [ ]:
# Load annotations
with open(TRAIN_ANNO, 'r') as f:
    train_data = json.load(f)

with open(VAL_ANNO, 'r') as f:
    val_data = json.load(f)

with open(LABEL_MAP, 'r') as f:
    label_map = json.load(f)

print(f"Train: {len(train_data['images'])} images, {len(train_data['annotations'])} objects")
print(f"Val:   {len(val_data['images'])} images, {len(val_data['annotations'])} objects")
print(f"Classes in label_map: {len(label_map)}")

# Count class instances
class_counts = Counter()
for ann in train_data['annotations']:
    class_counts[ann['category_id']] += 1

print(f"\nTraining classes: {len(class_counts)}")
print(f"Max objects per class: {max(class_counts.values())}")
print(f"Min objects per class:  {min(class_counts.values())}")

# Select the 30 most frequent classes
TOP_N = 30
MAX_IMAGES_PER_CLASS = 1000

top_classes   = {cat_id for cat_id, _ in class_counts.most_common(TOP_N)}
cat_id_to_new = {old: new for new, old in enumerate(sorted(top_classes), start=1)}
NUM_CLASSES   = len(top_classes) + 1
print(f"\nTop {TOP_N} classes: {sorted(top_classes)}")
print(f"NUM_CLASSES (including background): {NUM_CLASSES}")

# Keep selected classes
train_anns_filtered = [a for a in train_data['annotations'] if a['category_id'] in top_classes]
val_anns_filtered   = [a for a in val_data['annotations']   if a['category_id'] in top_classes]

train_img_ids = {a['image_id'] for a in train_anns_filtered}
val_img_ids   = {a['image_id'] for a in val_anns_filtered}

train_imgs_filtered = [img for img in train_data['images'] if img['id'] in train_img_ids]
val_imgs_filtered   = [img for img in val_data['images']   if img['id'] in val_img_ids]

print(f"\nAfter selecting top {TOP_N}:")
print(f"Train: {len(train_imgs_filtered)} images, {len(train_anns_filtered)} objects")

# Count before balancing
class_counts_top = Counter()
for ann in train_anns_filtered:
    class_counts_top[ann['category_id']] += 1

# Limit each class to 1,000 images
from collections import defaultdict

cat_to_imgs = defaultdict(list)
for ann in train_anns_filtered:
    cat_to_imgs[ann['category_id']].append(ann['image_id'])

imgs_to_keep = set()
for cat_id, img_ids in cat_to_imgs.items():
    unique_imgs = list(dict.fromkeys(img_ids))
    if len(unique_imgs) >= MAX_IMAGES_PER_CLASS:
        imgs_to_keep.update(unique_imgs[:MAX_IMAGES_PER_CLASS])
    else:
        imgs_to_keep.update(unique_imgs)

train_imgs_filtered = [img for img in train_imgs_filtered if img['id'] in imgs_to_keep]
train_anns_filtered = [a   for a   in train_anns_filtered if a['image_id'] in imgs_to_keep]

print(f"\nAfter filtering (up to {MAX_IMAGES_PER_CLASS} images per class):")
print(f"Train: {len(train_imgs_filtered)} images, {len(train_anns_filtered)} objects")

# Count final class instances
class_counts_final = Counter()
for ann in train_anns_filtered:
    class_counts_final[ann['category_id']] += 1

# Plot class distribution
sorted_final = sorted(class_counts_final.items(), key=lambda x: -x[1])

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar([str(x[0]) for x in sorted_final], [x[1] for x in sorted_final], color='darkorange')
ax.axhline(y=MAX_IMAGES_PER_CLASS, color='red', linestyle='--', label=f'Max images per class = {MAX_IMAGES_PER_CLASS}')
ax.set_title(f'Filtered RTSD dataset — top {TOP_N} classes', fontsize=13)
ax.set_xlabel('category_id')
ax.set_ylabel('Object count')
ax.tick_params(axis='x', rotation=45)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('class_distribution_final.png', dpi=100)
plt.show()

## 4. Dataset class

In [ ]:
class RTSDDataset(Dataset):
    def __init__(self, images, annotations, frames_dir, cat_id_to_new, transforms=None):
        """Load RTSD images, bounding boxes, and remapped class labels."""
        self.images      = images
        self.frames_dir  = frames_dir
        self.cat_id_to_new = cat_id_to_new
        self.transforms  = transforms

        # Index annotations by image
        self.img_to_anns = defaultdict(list)
        for ann in annotations:
            self.img_to_anns[ann['image_id']].append(ann)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_id   = img_info['id']

        # Use the filename without the original directory
        fname    = os.path.basename(img_info['file_name'])
        img_path = os.path.join(self.frames_dir, fname)
        img      = Image.open(img_path).convert('RGB')

        anns = self.img_to_anns[img_id]

        boxes, labels = [], []
        for ann in anns:
            new_cat = self.cat_id_to_new.get(ann['category_id'])
            if new_cat is None:
                continue
            # COCO bbox: [x_min, y_min, width, height] -> [x1, y1, x2, y2]
            x, y, w, h = ann['bbox']
            x2, y2 = x + w, y + h
            # Skip invalid boxes
            if w <= 0 or h <= 0:
                continue
            boxes.append([x, y, x2, y2])
            labels.append(new_cat)

        if len(boxes) == 0:
            boxes  = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros(0, dtype=torch.int64)
        else:
            boxes  = torch.as_tensor(boxes,  dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        area    = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) if len(boxes) > 0 else torch.zeros(0)
        iscrowd = torch.zeros(len(labels), dtype=torch.int64)

        target = {
            'boxes':    boxes,
            'labels':   labels,
            'image_id': torch.tensor([img_id]),
            'area':     area,
            'iscrowd':  iscrowd,
        }

        # Convert image to tensor
        img = F.to_tensor(img)

        if self.transforms:
            img, target = self.transforms(img, target)

        return img, target


def collate_fn(batch):
    return tuple(zip(*batch))


print("RTSDDataset is ready.")

## 5. Data loaders

In [ ]:
import random

# Split filtered images into train/val/test
random.seed(42)
all_imgs = train_imgs_filtered.copy()
random.shuffle(all_imgs)

n_test = int(len(all_imgs) * 0.15)
n_val  = int(len(all_imgs) * 0.15)

test_imgs_split  = all_imgs[:n_test]
val_imgs_split   = all_imgs[n_test : n_test + n_val]
train_imgs_split = all_imgs[n_test + n_val:]

print(f"Train: {len(train_imgs_split)} | Val: {len(val_imgs_split)} | Test: {len(test_imgs_split)}")

# Select annotations for each split
train_ids = {img['id'] for img in train_imgs_split}
val_ids   = {img['id'] for img in val_imgs_split}
test_ids  = {img['id'] for img in test_imgs_split}

train_anns_split = [a for a in train_anns_filtered if a['image_id'] in train_ids]
val_anns_split   = [a for a in train_anns_filtered if a['image_id'] in val_ids]
test_anns_split  = [a for a in train_anns_filtered if a['image_id'] in test_ids]

# Create datasets
dataset_train = RTSDDataset(train_imgs_split, train_anns_split, FRAMES_DIR, cat_id_to_new)
dataset_val   = RTSDDataset(val_imgs_split,   val_anns_split,   FRAMES_DIR, cat_id_to_new)
dataset_test  = RTSDDataset(test_imgs_split,  test_anns_split,  FRAMES_DIR, cat_id_to_new)

data_loader_train = DataLoader(dataset_train, batch_size=2, shuffle=True,  collate_fn=collate_fn, num_workers=0)
data_loader_val   = DataLoader(dataset_val,   batch_size=2, shuffle=False, collate_fn=collate_fn, num_workers=0)
data_loader_test  = DataLoader(dataset_test,  batch_size=2, shuffle=False, collate_fn=collate_fn, num_workers=0)

print(f"Train batches: {len(data_loader_train)}")
print(f"Val batches:   {len(data_loader_val)}")
print(f"Test batches:  {len(data_loader_test)}")

## 6. Modified Faster R-CNN (COCO → GroupNorm + SiLU)

Build ResNet50-FPN, replace the normalization and activation layers, and load compatible COCO-pretrained convolution/FPN weights. GroupNorm is initialized separately, as are the RPN and detection head. The next cell verifies the layer types when executed.

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.faster_rcnn import fasterrcnn_resnet50_fpn
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.ops import MultiScaleRoIAlign
from torchvision.ops.misc import FrozenBatchNorm2d

# Replace normalization and activation layers.
def replace_bn_relu(module: nn.Module, num_groups: int = 32) -> nn.Module:
    """Replace BatchNorm/FrozenBatchNorm with GroupNorm and ReLU with SiLU."""
    for name, child in list(module.named_children()):
        if isinstance(child, (nn.BatchNorm2d, FrozenBatchNorm2d)):
            num_channels = child.num_features
            groups = num_groups
            while num_channels % groups != 0 and groups > 1:
                groups //= 2
            setattr(module, name, nn.GroupNorm(
                num_groups=groups,
                num_channels=num_channels
            ))
        elif isinstance(child, nn.ReLU):
            setattr(module, name, nn.SiLU(inplace=False))
        else:
            replace_bn_relu(child, num_groups)
    return module

# Load COCO weights, excluding original normalization parameters.
print('Loading COCO-pretrained Faster R-CNN...')
coco_model = fasterrcnn_resnet50_fpn(weights='DEFAULT')

old_norm_prefixes = tuple(
    f'{name}.' for name, layer in coco_model.backbone.named_modules()
    if isinstance(layer, (nn.BatchNorm2d, FrozenBatchNorm2d))
)
coco_backbone_state = {
    key: value for key, value in coco_model.backbone.state_dict().items()
    if not key.startswith(old_norm_prefixes)
}
print(f'Backbone/FPN state keys to load: {len(coco_backbone_state)}')

del coco_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Build and modify the backbone.
backbone = resnet_fpn_backbone(
    backbone_name='resnet50',
    weights=None,
    trainable_layers=5,
)
backbone = replace_bn_relu(backbone, num_groups=32)

# Verify normalization and activation layers.
num_gn = sum(isinstance(layer, nn.GroupNorm) for layer in backbone.modules())
num_bn = sum(isinstance(layer, (nn.BatchNorm2d, FrozenBatchNorm2d))
             for layer in backbone.modules())
num_silu = sum(isinstance(layer, nn.SiLU) for layer in backbone.modules())
print(f'GroupNorm: {num_gn} | remaining BatchNorm/FrozenBatchNorm: {num_bn} | SiLU: {num_silu}')
if num_gn == 0 or num_bn != 0 or num_silu == 0:
    raise RuntimeError('Backbone does not match GroupNorm + SiLU')

# Load matching COCO weights; keep GroupNorm initialized separately.
missing, unexpected = backbone.load_state_dict(coco_backbone_state, strict=False)
expected_missing = {
    f'{name}.{param}'
    for name, layer in backbone.named_modules()
    if isinstance(layer, nn.GroupNorm)
    for param in ('weight', 'bias')
}
if set(missing) != expected_missing or unexpected:
    raise RuntimeError(
        f'Check COCO weight loading: missing={missing}, unexpected={unexpected}'
    )
print('COCO backbone/FPN weights loaded; GroupNorm initialized separately.')

# Assemble the detector.
anchor_generator = AnchorGenerator(
    sizes=((32,), (64,), (128,), (256,), (512,)),
    aspect_ratios=((0.5, 1.0, 2.0),) * 5,
)
roi_pooler = MultiScaleRoIAlign(
    featmap_names=['0', '1', '2', '3'],
    output_size=7,
    sampling_ratio=2,
)
model = FasterRCNN(
    backbone=backbone,
    num_classes=NUM_CLASSES,  # 30 sign classes + background
    rpn_anchor_generator=anchor_generator,
    box_roi_pool=roi_pooler,
)
model.to(device)

# Report model size.
print(f'Model initialized on {device}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'NUM_CLASSES: {NUM_CLASSES} (including background)')


## 7. Optimizer and learning-rate scheduler

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.SGD(
    params,
    lr           = LR,
    momentum     = MOMENTUM,
    weight_decay = WEIGHT_DECAY,
)

# Decay learning rate at epochs 8 and 12
lr_scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones = [8, 12],
    gamma      = 0.1,
)

print(f"Optimizer: SGD | lr={LR} | momentum={MOMENTUM} | weight_decay={WEIGHT_DECAY}")
print(f"LR scheduler: MultiStepLR | milestones=[8, 12] | gamma=0.1")

## 8. COCO evaluation utilities

Load the torchvision detection reference utilities if they are not available locally.

In [ ]:
import urllib.request, os

BASE = 'https://raw.githubusercontent.com/pytorch/vision/main/references/detection/'
for fname in ['engine.py', 'coco_eval.py', 'coco_utils.py', 'utils.py']:
    if not os.path.exists(fname):
        urllib.request.urlretrieve(BASE + fname, fname)
        print(f'Downloaded: {fname}')
    else:
        print(f'Already exists: {fname}')

from engine import train_one_epoch, evaluate
print('engine.py imported.')

## 9. Training and validation

In [ ]:
# Train losses
losses_hist, loss_box_reg, loss_rpn_box_reg, loss_classifier, loss_objectness = [], [], [], [], []

# Val losses
val_losses, val_loss_box_reg, val_loss_rpn_box_reg, val_loss_classifier, val_loss_objectness = [], [], [], [], []

# Track 12 COCO AP/AR metrics
stat0,stat1,stat2,stat3,stat4,stat5,stat6,stat7,stat8,stat9,stat10,stat11 = [],[],[],[],[],[],[],[],[],[],[],[]

best_val_loss = float('inf')

print(f"Training: {NUM_EPOCHS} epochs | device: {device}")
print("=" * 60)

for epoch in range(NUM_EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")

    # --- Train ---
    metrics = train_one_epoch(model, optimizer, data_loader_train, device, epoch, print_freq=50)
    print(f"\nmetrics")
    losses_hist.append(float(str(metrics.meters['loss']).split(' ')[0]))
    print(f"\nlosses_hist")
    loss_box_reg.append(float(str(metrics.meters['loss_box_reg']).split(' ')[0]))
    loss_rpn_box_reg.append(float(str(metrics.meters['loss_rpn_box_reg']).split(' ')[0]))
    loss_classifier.append(float(str(metrics.meters['loss_classifier']).split(' ')[0]))
    loss_objectness.append(float(str(metrics.meters['loss_objectness']).split(' ')[0]))
    print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
    lr_scheduler.step()
    print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
    # Validation COCO metrics
    coco_evaluator, _ = evaluate(model, data_loader_val, device=device)
    stat = coco_evaluator.coco_eval['bbox'].stats
    for i, lst in enumerate([stat0,stat1,stat2,stat3,stat4,stat5,stat6,stat7,stat8,stat9,stat10,stat11]):
        lst.append(stat[i])

    # --- Val losses ---
    model.train()
    vl, vlb, vlrpn, vlc, vlo = 0, 0, 0, 0, 0
    with torch.no_grad():
        for imgs, targets in data_loader_val:
            imgs    = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(imgs, targets)
            vl    += sum(loss_dict.values()).item()
            vlb   += loss_dict['loss_box_reg'].item()
            vlrpn += loss_dict['loss_rpn_box_reg'].item()
            vlc   += loss_dict['loss_classifier'].item()
            vlo   += loss_dict['loss_objectness'].item()
    model.eval()

    n = len(data_loader_val)
    val_losses.append(vl / n)
    val_loss_box_reg.append(vlb / n)
    val_loss_rpn_box_reg.append(vlrpn / n)
    val_loss_classifier.append(vlc / n)
    val_loss_objectness.append(vlo / n)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"train loss: {losses_hist[-1]:.4f} | "
          f"val loss: {val_losses[-1]:.4f} | "
          f"AP@0.5: {stat1[-1]:.3f} | "
          f"AP@0.5:0.95: {stat0[-1]:.3f}")

    # Save the lowest-validation-loss checkpoint
    if val_losses[-1] < best_val_loss:
        best_val_loss = val_losses[-1]
        torch.save(model.state_dict(), 'best_model_rtsd.pth')
        print(f"  ✓ Best model saved (val_loss={best_val_loss:.4f})")

print("\n" + "=" * 60)
print("Training complete!")

## 10. Final test-set evaluation

In [ ]:
model.load_state_dict(torch.load('model_modernized.pth', map_location=device))
model.to(device)

print("Final TEST evaluation (best model):")
coco_evaluator_final, _ = evaluate(model, data_loader_test, device=device)
final_stats = coco_evaluator_final.coco_eval['bbox'].stats

metrics_names = [
    'AP @IoU=0.50:0.95', 'AP @IoU=0.50', 'AP @IoU=0.75',
    'AP small',          'AP medium',    'AP large',
    'AR @maxDets=1',     'AR @maxDets=10','AR @maxDets=100',
    'AR small',          'AR medium',    'AR large',
]
print("\nMetrics:")
for name, val in zip(metrics_names, final_stats):
    print(f"  {name:25s}: {val:.4f}")

In [ ]:
import cv2
import numpy as np
from torchvision import transforms

# Load the saved model
model.load_state_dict(torch.load('model_modernized.pth', map_location=device))
model.to(device)

def add_gaussian_noise(image, intensity=25):
    noise = np.random.normal(0, intensity, image.shape)
    noisy = image + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)

def add_salt_pepper_noise(image, prob=0.02):
    noisy = image.copy()
    salt   = np.random.random(image.shape[:2]) < prob/2
    pepper = np.random.random(image.shape[:2]) < prob/2
    noisy[salt]   = 255
    noisy[pepper] = 0
    return noisy

class NoisyRTSDDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, noise_type='gaussian'):
        self.dataset    = dataset
        self.noise_type = noise_type
        self.to_tensor  = transforms.ToTensor()

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img_tensor, target = self.dataset[idx]
        # Convert to NumPy
        img_np = (img_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        # Add noise
        if self.noise_type == 'gaussian':
            img_np = add_gaussian_noise(img_np, intensity=25)
        elif self.noise_type == 'salt_pepper':
            img_np = add_salt_pepper_noise(img_np, prob=0.02)
        # Convert back to tensor
        img_tensor = self.to_tensor(img_np)
        return img_tensor, target

# Create noisy test loaders
loader_gaussian = DataLoader(
    NoisyRTSDDataset(dataset_test, noise_type='gaussian'),
    batch_size=2, shuffle=False, collate_fn=collate_fn, num_workers=0
)
loader_saltpepper = DataLoader(
    NoisyRTSDDataset(dataset_test, noise_type='salt_pepper'),
    batch_size=2, shuffle=False, collate_fn=collate_fn, num_workers=0
)

# Clean-image metrics are stored in final_stats
print("Gaussian-noise evaluation:")
eval_gaussian, _ = evaluate(model, loader_gaussian, device=device)
stats_gaussian   = eval_gaussian.coco_eval['bbox'].stats

print("\nSalt-and-pepper evaluation:")
eval_sp, _ = evaluate(model, loader_saltpepper, device=device)
stats_sp   = eval_sp.coco_eval['bbox'].stats

metrics_names = [
    'AP @IoU=0.50:0.95', 'AP @IoU=0.50', 'AP @IoU=0.75',
    'AP small',          'AP medium',    'AP large',
    'AR @maxDets=1',     'AR @maxDets=10','AR @maxDets=100',
    'AR small',          'AR medium',    'AR large',
]

print("\nMetric comparison:")
print(f"{'Metric':<25} {'Clean':>10} {'Gaussian':>10} {'Salt&Pepper':>12}")
print("-" * 60)
for name, clean, gauss, sp in zip(metrics_names, final_stats, stats_gaussian, stats_sp):
    print(f"{name:<25} {clean:>10.4f} {gauss:>10.4f} {sp:>12.4f}")